# Volume 2, Chapter 3: Group Actions and Counting (Python companion)

Exercises first, solutions at the end. We test Burnside's lemma against brute force, count colourings of the cube,
count graphs up to isomorphism with Pólya's theorem, measure how often two elements commute, and look at the
computer-science side of counting up to symmetry: **canonical forms**, and why picking a random object is not the same
as picking a random orbit.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools
import math
import random
from collections import Counter
from fractions import Fraction
import networkx as nx

## Exercises

**P3.1 (★) Necklaces two ways.** Count the $k$-colour necklaces with $n$ beads (rotations only) by brute force, using the lexicographically smallest rotation as a canonical form, and with the formula $N(n,k)=\frac1n\sum_{d\mid n}\varphi(d)k^{n/d}$ of Example 3.8. Compare for $n\le 10$, $k\le 3$. *Guess first: is $N(n,2)$ close to $2^n/n$?*

In [ ]:
# Your code here


**P3.2 (★) The cube.** Build the $24$ rotations of the cube as permutations of its $6$ faces, starting from two quarter turns and closing under composition. Check the cycle types used in Exercise 3.1, count $3$-colourings of the faces up to rotation by brute force, and count those with exactly two red faces with Pólya's weighted form (Theorem 3.7).

In [ ]:
# Your code here


**P3.3 (★★) Counting graphs.** A graph on $n$ labelled vertices is a subset of the $\binom n2$ pairs, and $\mathfrak S_n$ acts on pairs. By Burnside the number of graphs up to isomorphism is $\frac1{n!}\sum_{\sigma}2^{c(\sigma)}$, where $c(\sigma)$ is the number of cycles of $\sigma$ on pairs, and $c(\sigma)$ depends only on the cycle type of $\sigma$. Compute the counts for $n\le 10$ by summing over partitions of $n$, and check $n\le 5$ by brute force (Exercise 3.7 gives $11$ for $n=4$).

*How to think about it:* for a cycle of length $a$ of $\sigma$, work out how many cycles $\sigma$ has on the pairs inside it; for two cycles of lengths $a$ and $b$, on the $ab$ pairs between them.

In [ ]:
# Your code here


**P3.4 (★★) Commuting probability.** The probability that two uniformly random elements of $G$ commute is $k(G)/|G|$, where $k(G)$ is the number of conjugacy classes (Exercise 3.8). Compute it for $\mathfrak S_3$, $D_4$, $Q_8$, $A_4$, $\mathfrak S_4$, $A_5$ and $\mathfrak S_5$, both from the class count and by Monte Carlo, and check that it never exceeds $5/8$ for a non-abelian group.

In [ ]:
# Your code here


**P3.5 (★★★) CS application: canonical forms and biased sampling.** A program that stores objects "up to symmetry" (molecules, game positions, graphs in a database) maps each object to a canonical representative of its orbit and stores that.
1. Deduplicate all binary strings of length $12$ up to rotation and check the count against $N(12,2)$.
2. Draw a random string, then take its canonical form: this picks an orbit with probability proportional to its size. Measure how far this is from the uniform distribution on necklaces, for $n=6$ (Figure 3.1 lists the orbit sizes).
3. Use the orbit–stabiliser theorem to fix it: accept a sampled string $s$ with probability $|\mathrm{Stab}(s)|/n$ (rejection sampling). Check that the result is uniform on orbits.

In [ ]:
# Your code here


---
## Solutions

### P3.1

In [ ]:
def canon(s):
    return min(s[i:] + s[:i] for i in range(len(s)))

def brute(n, k):
    return len({canon(s) for s in itertools.product(range(k), repeat=n)})

def phi(d):
    return sum(1 for i in range(1, d + 1) if math.gcd(i, d) == 1)

def formula(n, k):
    return sum(phi(d) * k ** (n // d) for d in range(1, n + 1) if n % d == 0) // n

for k in (2, 3):
    print(f"k={k}:", [(brute(n, k), formula(n, k)) for n in range(1, 11)])
print("N(6,2) =", formula(6, 2), " N(6,3) =", formula(6, 3))
print("N(n,2) / (2^n/n) for n=10,20,30:", [round(formula(n, 2) / (2**n / n), 4) for n in (10, 20, 30)])

Brute force and formula agree, and $N(6,3)=130$ as in the text. The ratio to $2^n/n$ tends to $1$: almost every orbit has the maximal size $n$, because only strings with a period dividing $n$ have a non-trivial stabiliser, and they are exponentially rare. Most of the identity term $k^n$ survives, and the others are of size at most $k^{n/2}$.

### P3.2

In [ ]:
# faces: 0=up 1=down 2=front 3=back 4=left 5=right; a permutation p sends face i to face p[i]
def compose(p, q):            # first q, then p
    return tuple(p[q[i]] for i in range(6))

turn_z = (0, 1, 5, 4, 2, 3)   # quarter turn about the up-down axis: front->right->back->left->front
turn_x = (3, 2, 0, 1, 4, 5)   # quarter turn about the left-right axis: up->back->down->front->up
G = {tuple(range(6))}
frontier = list(G)
while frontier:
    g = frontier.pop()
    for t in (turn_z, turn_x):
        h = compose(t, g)
        if h not in G:
            G.add(h); frontier.append(h)
print("number of rotations:", len(G))

def cycle_type(p):
    seen, lengths = set(), []
    for i in range(len(p)):
        if i not in seen:
            l, j = 0, i
            while j not in seen:
                seen.add(j); j = p[j]; l += 1
            lengths.append(l)
    return tuple(sorted(lengths, reverse=True))

print(Counter(cycle_type(g) for g in G))

orbits = {min(tuple(c[g[i]] for i in range(6)) for g in G) for c in itertools.product(range(3), repeat=6)}
print("3-colourings up to rotation (brute force):", len(orbits))

# Polya weighted: substitute p_l = r^l + b^l + g^l and read off the coefficient of r^2
import sympy as sp
r, b, gg = sp.symbols('r b g')
Z = sum(sp.Mul(*[(r**l + b**l + gg**l) for l in cycle_type(g)]) for g in G) / 24
poly = sp.Poly(sp.expand(Z), r)
two_red = sp.expand(poly.coeff_monomial(r**2)).subs({b: 1, gg: 1})
print("exactly two red faces:", two_red,
      " brute force:", sum(1 for o in orbits if o.count(0) == 2))

The cycle types are exactly those of Exercise 3.1: $1$ identity of type $1^6$, $6$ quarter turns $4\,1^2$, $3$ face half-turns $2^2 1^2$, $8$ third turns $3^2$, and $6$ edge half-turns $2^3$. Two red faces up to rotation: the two red faces are opposite or adjacent ($2$ ways), and the other four faces are coloured with blue and green; the weighted count gives $16$, confirmed by brute force. By hand: with the red faces opposite, the four side faces form a band on which the stabiliser acts as $D_4$, giving $6$ colourings; with the red faces adjacent, the stabiliser is the half turn about the edge axis between them, which swaps the remaining faces in two pairs, so Burnside gives $(2^4+2^2)/2=10$. Total $16$.

### P3.3

In [ ]:
def partitions(n, largest=None):
    largest = largest or n
    if n == 0:
        yield []
        return
    for a in range(min(n, largest), 0, -1):
        for rest in partitions(n - a, a):
            yield [a] + rest

def class_size(lam):
    n = sum(lam)
    denom = 1
    for a, m in Counter(lam).items():
        denom *= a**m * math.factorial(m)
    return math.factorial(n) // denom

def cycles_on_pairs(lam):
    c = 0
    for a in lam:                       # pairs inside one cycle of length a
        c += a // 2                     # (a-1)/2 if a odd; (a-2)/2 + 1 if a even
    for i in range(len(lam)):           # pairs between two cycles: gcd(a,b) cycles of length lcm(a,b)
        for j in range(i + 1, len(lam)):
            c += math.gcd(lam[i], lam[j])
    return c

def graphs_up_to_iso(n):
    return sum(class_size(l) * 2 ** cycles_on_pairs(l) for l in partitions(n)) // math.factorial(n)

print([graphs_up_to_iso(n) for n in range(1, 11)])

def brute_graphs(n):
    pairs = list(itertools.combinations(range(n), 2))
    perms = list(itertools.permutations(range(n)))
    seen = set()
    for mask in range(2 ** len(pairs)):
        E = [pairs[i] for i in range(len(pairs)) if mask >> i & 1]
        seen.add(min(tuple(sorted(tuple(sorted((p[u], p[v]))) for u, v in E)) for p in perms))
    return len(seen)

print([brute_graphs(n) for n in range(1, 6)])
print("networkx atlas (graphs with up to 7 vertices):",
      Counter(G.number_of_nodes() for G in nx.graph_atlas_g()))

Inside a cycle of length $a$, the $\binom a2$ pairs fall into cycles of length $a$, except, when $a$ is even, the $a/2$ "diameters" $\{i,i+a/2\}$, which form one cycle of length $a/2$; in both cases that is $\lfloor a/2\rfloor$ cycles. Between cycles of lengths $a$ and $b$, the $ab$ pairs form cycles of length $\operatorname{lcm}(a,b)$, so there are $ab/\operatorname{lcm}(a,b)=\gcd(a,b)$ of them.

The sequence $1,2,4,11,34,156,1044,12346,274668,12005168$ (OEIS A000088) agrees with brute force for $n\le5$ and with the networkx graph atlas for $n\le7$. For large $n$ the identity term dominates and the count is close to $2^{\binom n2}/n!$: almost every graph has no symmetry at all (Erdős–Rényi, 1963).

### P3.4

In [ ]:
def perm_group(gens):
    n = len(gens[0])
    e = tuple(range(n))
    G = {e}; frontier = [e]
    while frontier:
        g = frontier.pop()
        for s in gens:
            h = tuple(s[g[i]] for i in range(n))
            if h not in G:
                G.add(h); frontier.append(h)
    return G

def mul(p, q):
    return tuple(p[q[i]] for i in range(len(p)))

def inv(p):
    r = [0] * len(p)
    for i, x in enumerate(p):
        r[x] = i
    return tuple(r)

def num_classes(G):
    left, k = set(G), 0
    while left:
        x = left.pop(); k += 1
        left -= {mul(mul(g, x), inv(g)) for g in G}
    return k

# Q8 = {±1, ±i, ±j, ±k}, acting on itself by left multiplication
units = ["1", "i", "j", "k"]
table = {("1", x): (1, x) for x in units} | {(x, "1"): (1, x) for x in units}
table |= {(x, x): (-1, "1") for x in "ijk"}
table |= {("i", "j"): (1, "k"), ("j", "k"): (1, "i"), ("k", "i"): (1, "j"),
          ("j", "i"): (-1, "k"), ("k", "j"): (-1, "i"), ("i", "k"): (-1, "j")}
elems = [(s, u) for s in (1, -1) for u in units]
def qmul(a, b):
    sign, u = table[(a[1], b[1])]
    return (a[0] * b[0] * sign, u)
left = lambda a: tuple(elems.index(qmul(a, b)) for b in elems)
Q8 = perm_group([left((1, "i")), left((1, "j"))])

groups = {
    "S3": perm_group([(1, 0, 2), (1, 2, 0)]),
    "D4": perm_group([(1, 2, 3, 0), (0, 3, 2, 1)]),
    "Q8": Q8,
    "A4": perm_group([(1, 2, 0, 3), (1, 0, 3, 2)]),
    "S4": perm_group([(1, 0, 2, 3), (1, 2, 3, 0)]),
    "A5": perm_group([(1, 2, 0, 3, 4), (1, 2, 3, 4, 0)]),
    "S5": perm_group([(1, 0, 2, 3, 4), (1, 2, 3, 4, 0)]),
}
random.seed(1)
for name, G in groups.items():
    L = list(G)
    k = num_classes(G)
    trials = 20000
    hits = sum(1 for _ in range(trials) if mul(*(pair := (random.choice(L), random.choice(L)))) == mul(pair[1], pair[0]))
    print(f"{name}: |G|={len(G):3d}  k(G)={k:2d}  k/|G|={str(Fraction(k, len(G))):6s} = {k/len(G):.3f}  Monte Carlo {hits/trials:.3f}")

$Q_8$ is realised by left multiplication on its own $8$ elements (Cayley's theorem 3.3), from the quaternion rules $i^2=j^2=k^2=-1$, $ij=k$. $D_4$ and $Q_8$ both have $5$ classes and reach the bound $5/8$ exactly, as Exercise 3.8 predicts for any group with $G/Z(G)\cong(\Z/2\Z)^2$. The simple group $A_5$ has $5$ classes for $60$ elements: two random elements commute only $1/12$ of the time. Gustafson's paper (cited in the chapter) is the source of the $5/8$ theorem.

### P3.5

In [ ]:
n = 12
reps = {canon(s) for s in itertools.product((0, 1), repeat=n)}
print(len(reps), formula(n, 2))

n = 6
def stab_size(s):
    return sum(1 for i in range(n) if s[i:] + s[:i] == s)

orbit_size = {}
for s in itertools.product((0, 1), repeat=n):
    orbit_size[canon(s)] = n // stab_size(s)
neck = sorted(orbit_size)
print("orbit sizes:", [orbit_size[c] for c in neck])

random.seed(2)
T = 140000
naive = Counter(canon(tuple(random.randint(0, 1) for _ in range(n))) for _ in range(T))
fixed = Counter()
while sum(fixed.values()) < T // 4:
    s = tuple(random.randint(0, 1) for _ in range(n))
    if random.random() < stab_size(s) / n:
        fixed[canon(s)] += 1
def tv(c):
    tot = sum(c.values())
    return 0.5 * sum(abs(c[x] / tot - 1 / len(neck)) for x in neck)
print(f"total variation from uniform: naive {tv(naive):.3f}, rejection {tv(fixed):.3f}")
print("naive frequency of 000000 vs 010101:", naive[(0,)*6] / T, naive[(0, 1)*3] / T, " uniform would be", round(1/14, 4))

Deduplicating by canonical form gives $N(12,2)=352$. The naive sampler picks the orbit of $s$ with probability $|\text{orbit}|/2^n$, so $000000$ (orbit size $1$) appears with probability $1/64$ instead of $1/14$: a total variation distance of about $0.2$. The rejection step accepts $s$ with probability $|\mathrm{Stab}(s)|/n=1/|\text{orbit of }s|$ (orbit–stabiliser); each orbit $O$ is then hit with probability $\frac{|O|}{2^n}\cdot\frac1{|O|}=\frac1{2^n}$ per draw, the same for every orbit, and the measured distance to uniform drops to sampling noise. (The tempting rule "accept with probability $1/|\mathrm{Stab}|$" is backwards: it makes the bias worse.) This is the orbit–stabiliser theorem turned into an algorithm, and a simple cousin of the "Burnside process" (Jerrum, 1993), a Markov chain that samples orbits uniformly when rejection would waste too many draws.